In [ ]:
pip install -r requirements.txt

In [66]:
import os
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from dotenv import load_dotenv
import json
from IPython.display import Markdown, display, update_display

In [67]:
load_dotenv()

True

In [68]:
import gradio as gr

In [69]:
openai = OpenAI()

In [70]:
system_prompt = "You are an AI Assistant"
def ask_gpt(prompt):
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":prompt}
    ]
    response = openai.chat.completions.create(
       model = "gpt-4o-mini",
        messages = messages,   
    )
    return response.choices[0].message.content

In [71]:
ask_gpt("tell a math joke")

"Why was the equal sign so humble?\n\nBecause it knew it wasn't less than or greater than anyone else!"

In [72]:
gr_interface = gr.Interface(
    fn=ask_gpt,
    inputs = [gr.Textbox(label="Ask me Anything", lines = 6)],
    outputs = [gr.Textbox(label="Response : ", lines = 10)],
    allow_flagging = "never"
)
gr_interface.launch()  #share=True (gives a public URL)

C:\Users\aswjayac\Desktop\python\LLM Eng\venv\lib\site-packages\gradio\interface.py:414: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [73]:
def stream_gpt(prompt):
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":prompt}
    ]
    stream = openai.chat.completions.create(
       model = "gpt-4o-mini",
        messages = messages,   
        stream = True
    )
    result = ""
    for chunks in stream:
        result += chunks.choices[0].delta.content or ""
        yield result   #if u yeild chunks , it replaces the old ones

In [74]:
gr_interface = gr.Interface(
    fn=stream_gpt,
    inputs = [gr.Textbox(label="Ask me Anything", lines = 6)],
    outputs = [gr.Markdown(label="Response : ")],
    allow_flagging = "never"
)
gr_interface.launch()  #share=True (gives a public URL)

C:\Users\aswjayac\Desktop\python\LLM Eng\venv\lib\site-packages\gradio\interface.py:414: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [ ]:
system_prompt = "You are an witty AI Assistant"
def chat(message,history):  #gradio chat sends input in this format, history = [(user asks , ass replies this),(),.....]
    hist = [{"role":"system","content":system_prompt}]
    for user_msg , assistant_msg in history:
        hist.append({"role":"user","content":user_msg})
        hist.append({"role":"assistant","content":assistant_msg})
    hist.append({"role":"user","content":message})

    stream = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = hist,   
        stream = True
    )
    result = ""
    for chunks in stream:
        result += chunks.choices[0].delta.content or ""
        yield result
        

In [ ]:
gr.ChatInterface(fn=chat).launch()

In [65]:
gr.close_all()

Closing server running on port: 7866
Closing server running on port: 7860
Closing server running on port: 7863
Closing server running on port: 7865
Closing server running on port: 7869
Closing server running on port: 7861
Closing server running on port: 7868
Closing server running on port: 7864
Closing server running on port: 7867
Closing server running on port: 7862


In [75]:
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool get_ticket_price called for {destination_city}")
    city = destination_city.lower()
    return ticket_prices.get(city, "Unknown")

In [76]:
price_tool = {
    "name":"get_ticket_price",
    "description":"Get the price of a return ticket to the destination city. Call this whenever you need to know the ticket price, for example when a customer asks 'How much is a ticket to this city'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [77]:
tools = [{"type": "function", "function": price_tool}]

In [78]:
system_message = "You are a helpful assistant for an Airline called FlightAI. "
system_message += "Give short, courteous answers, no more than 1 sentence. "
system_message += "Always be accurate. If you don't know the answer, say so."


def chat(new_cht_message,history):  #gradio chat sends input in this format, history = [(user asks , ass replies this),(),.....]
    msg_hist = [{"role":"system","content":system_message}] + history + [{"role":"user","content":new_cht_message}]
    print(msg_hist)
    #call openai to get which tool should i use and their params
    response = openai.chat.completions.create( model = "gpt-4o-mini",messages = msg_hist,tools = tools)
    
    #response has info on which tool u gonna , sends the name and the parameters. , we have to manully call the function
    # tools are not automatically called.
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message  #ChatCompletionMessage object
        tool_response, city = handle_tool_call(message)
        #ChatCompletionMessage - response from first call to LLM and the tool response with tool_call_id is added to history
        # next llm call will understand the content and gives the final response
        msg_hist.append(message)
        msg_hist.append(tool_response)
        print(msg_hist)
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=msg_hist)
    
    return response.choices[0].message.content

In [79]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    # message will be like below
    # ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_d0p1RGUr1rV0PafGJUsabNSq', function=Function(arguments='{"destination_city":"Paris"}', name='get_ticket_price'), type='function')])
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    city = arguments.get('destination_city')
    price = get_ticket_price(city) 
    #call it manually , have a dict X with {tool:tool_function} , then call X[tool_call.function.name](city) , to handle multiple tools
    response = {
        "role": "tool",
        "content": json.dumps({"destination_city": city,"price": price}),
        "tool_call_id": tool_call.id
    }
    return response, city

In [80]:
gr.ChatInterface(fn=chat,type="messages").launch()
#type messages , returns chat history in LLM format.

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


[{'role': 'system', 'content': "You are a helpful assistant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so."}, {'role': 'user', 'content': 'flight to berlin'}]
Tool get_ticket_price called for Berlin
[{'role': 'system', 'content': "You are a helpful assistant for an Airline called FlightAI. Give short, courteous answers, no more than 1 sentence. Always be accurate. If you don't know the answer, say so."}, {'role': 'user', 'content': 'flight to berlin'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Dq8Tr0Utv8IR9cc8Zs6jhuqs', function=Function(arguments='{"destination_city":"Berlin"}', name='get_ticket_price'), type='function')]), {'role': 'tool', 'content': '{"destination_city": "Berlin", "price": "$499"}', 'tool_call_id': 'call_Dq8Tr0Utv8IR9cc8Zs6jhuqs'}]
[{'rol

In [81]:
def chat(history): #this fn receives history , which also has the last user input
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    image = None
    
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response, city = handle_tool_call(message)
        messages.append(message)
        messages.append(response)
        image = artist(city)

        # tool_call_id json and chatcompletetionmessage object is shared when calling to LLM again
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        
    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]  #AI Response is added and sent to gradio Chatbox

    # Comment out or delete the next line if you'd rather skip Audio for now..
    talker(reply)
    
    return history, image #entire history
    # return {"role": "assistant", "content": reply}, image -- only send last response, history mainter by gr?


In [ ]:
with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500)
    with gr.Row():
        entry = gr.Textbox(label="Chat with our AI Assistant:")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, image_output]
    )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True)